# Step 6: Generate Publication-Quality Graphs, Qualitative Visual Results & Physical Area Quantification

Now that training has completed with **Best Validation F1: 99.40%**, this cell generates:
1. **Training & Convergence Curves (Figure 1)**: Loss (Train vs. Val) and Accuracy Metrics (F1 & IoU across all 60 epochs) extracted directly from `checkpoint_latest.pth`.
2. **Qualitative Change Detection Visuals (Figure 2)**: 5-column visualization showing $T_1$ (2006), $T_2$ (2012), Ground Truth, GMFRMNet Prediction, and False-Color Overlay.
3. **Physical Area Dynamics Report (Table 1)**: Real-world surface quantification ($m^2$, Hectares, $\text{km}^2$) using $0.5\text{ m}$ GSD and parcel connected component analysis.

All plots and reports are automatically saved to both `/content/figures` and `/content/drive/MyDrive/HRSCD_cache/figures`.

In [ ]:
import os
import json
import glob
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

# Setup directories
DRIVE_CACHE_DIR = "/content/drive/MyDrive/HRSCD_cache"
LOCAL_FIG_DIR = "/content/figures"
DRIVE_FIG_DIR = os.path.join(DRIVE_CACHE_DIR, "figures")
os.makedirs(LOCAL_FIG_DIR, exist_ok=True)
os.makedirs(DRIVE_FIG_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = os.path.join(DRIVE_CACHE_DIR, "checkpoint_latest.pth")
BEST_MODEL_PATH = os.path.join(DRIVE_CACHE_DIR, "best_gmfrmnet.pth")
PATCHES_DIR = "/content/dataset_patches"
SPLIT_CACHE_PATH = os.path.join(DRIVE_CACHE_DIR, "train_val_split.json")

print(f"Using compute device: {device}")

# =====================================================================
# 1. PLOT PUBLICATION-QUALITY TRAINING CURVES (LOSS & METRICS)
# =====================================================================
print("\n--- 1. Plotting Training and Validation Curves ---")
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
    history = ckpt.get("history", {})
    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])
    val_f1 = history.get("val_f1", [])
    val_iou = history.get("val_iou", [])
    epochs = range(1, len(train_loss) + 1)
    
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 11,
        "axes.linewidth": 1.2,
        "grid.alpha": 0.4,
    })
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=300)
    
    # Subplot A: Loss Curves
    ax1.plot(epochs, train_loss, color="#1f77b4", linewidth=2.2, label="Training Loss (Adaptive)")
    ax1.plot(epochs, val_loss, color="#d62728", linewidth=2.2, linestyle="--", label="Validation Loss")
    ax1.set_title("(a) Convergence Behavior Across Epochs", fontweight="bold", pad=12)
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(frameon=True, facecolor="white", edgecolor="#ccc")
    ax1.grid(True, linestyle=":")
    ax1.set_xlim(1, len(train_loss))
    
    # Subplot B: Accuracy Metrics (IoU & F1)
    ax2.plot(epochs, val_f1, color="#2ca02c", linewidth=2.2, label=f"Validation F1-Score (Peak: {max(val_f1):.2f}%)")
    ax2.plot(epochs, val_iou, color="#ff7f0e", linewidth=2.2, linestyle="-.", label=f"Validation IoU (Peak: {max(val_iou):.2f}%)")
    ax2.set_title("(b) Accuracy Progression (F1 & IoU)", fontweight="bold", pad=12)
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Percentage (%)")
    ax2.set_ylim(min(min(val_f1), min(val_iou)) - 5, 102)
    ax2.legend(frameon=True, facecolor="white", edgecolor="#ccc")
    ax2.grid(True, linestyle=":")
    ax2.set_xlim(1, len(train_loss))
    
    plt.tight_layout()
    loss_fig_png = os.path.join(LOCAL_FIG_DIR, "training_metrics_curve.png")
    loss_fig_pdf = os.path.join(LOCAL_FIG_DIR, "training_metrics_curve.pdf")
    plt.savefig(loss_fig_png, bbox_inches="tight")
    plt.savefig(loss_fig_pdf, bbox_inches="tight")
    plt.savefig(os.path.join(DRIVE_FIG_DIR, "training_metrics_curve.png"), bbox_inches="tight")
    plt.savefig(os.path.join(DRIVE_FIG_DIR, "training_metrics_curve.pdf"), bbox_inches="tight")
    plt.show()
    print(f"Saved curve figures to: {loss_fig_png} and Drive!")
else:
    print("Warning: checkpoint_latest.pth not found.")

# =====================================================================
# 2. LOAD TRAINED MODEL FOR INFERENCE
# =====================================================================
# Ensure architecture classes are loaded in runtime
model = GMFRMNet(in_chans=3, num_classes=1).to(device)
if os.path.exists(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    print(f"Loaded best model weights from: {BEST_MODEL_PATH}")
elif os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device)["model_state"])
    print(f"Loaded latest weights from: {CHECKPOINT_PATH}")
model.eval()

# =====================================================================
# 3. QUALITATIVE VISUAL RESULTS (5-COLUMN FIGURE FOR PAPER)
# =====================================================================
print("\n--- 2. Generating Qualitative Visual Comparison Figure ---")
with open(SPLIT_CACHE_PATH, "r") as f:
    val_files = json.load(f)["val"]

# Select 4 representative patches with diverse change patterns
chosen_samples = []
for fn in val_files:
    lbl_p = os.path.join(PATCHES_DIR, "train", "label", fn)
    lbl_img = cv2.imread(lbl_p, cv2.IMREAD_GRAYSCALE)
    if lbl_img is not None:
        change_fraction = np.mean(lbl_img > 127)
        if 0.05 < change_fraction < 0.60:
            chosen_samples.append(fn)
    if len(chosen_samples) == 4:
        break

if len(chosen_samples) < 4:
    chosen_samples = val_files[:4]

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
fig, axes = plt.subplots(len(chosen_samples), 5, figsize=(15, 3.2 * len(chosen_samples)), dpi=300)
col_titles = ["(a) Time 1 (2006)", "(b) Time 2 (2012)", "(c) Ground Truth", "(d) GMFRM-AreaNet", "(e) Change Overlay"]

for i, fn in enumerate(chosen_samples):
    t1_p = os.path.join(PATCHES_DIR, "train", "t1", fn)
    t2_p = os.path.join(PATCHES_DIR, "train", "t2", fn)
    lbl_p = os.path.join(PATCHES_DIR, "train", "label", fn)
    
    t1_raw = Image.open(t1_p).convert("RGB").resize((256, 256))
    t2_raw = Image.open(t2_p).convert("RGB").resize((256, 256))
    lbl_raw = Image.open(lbl_p).convert("L").resize((256, 256))
    
    t1_tensor = normalize(T.ToTensor()(t1_raw)).unsqueeze(0).to(device)
    t2_tensor = normalize(T.ToTensor()(t2_raw)).unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred = model(t1_tensor, t2_tensor)
        prob = torch.sigmoid(pred).squeeze().cpu().numpy()
        pred_bin = (prob > 0.5).astype(np.uint8) * 255
        
    gt_bin = (np.array(lbl_raw) > 127).astype(np.uint8) * 255
    
    # Create overlay: Red highlight on T2
    t2_np = np.array(t2_raw)
    overlay = t2_np.copy()
    overlay[pred_bin > 0] = [230, 20, 20]  # Vibrant Red
    overlay_blend = cv2.addWeighted(t2_np, 0.55, overlay, 0.45, 0)
    
    row_imgs = [t1_raw, t2_raw, gt_bin, pred_bin, overlay_blend]
    
    for col in range(5):
        ax = axes[i, col] if len(chosen_samples) > 1 else axes[col]
        if col in [2, 3]:
            ax.imshow(row_imgs[col], cmap="gray")
        else:
            ax.imshow(row_imgs[col])
        if i == 0:
            ax.set_title(col_titles[col], fontweight="bold", pad=10, fontsize=12)
        ax.axis("off")

plt.tight_layout()
qual_png = os.path.join(LOCAL_FIG_DIR, "qualitative_comparison.png")
qual_pdf = os.path.join(LOCAL_FIG_DIR, "qualitative_comparison.pdf")
plt.savefig(qual_png, bbox_inches="tight")
plt.savefig(qual_pdf, bbox_inches="tight")
plt.savefig(os.path.join(DRIVE_FIG_DIR, "qualitative_comparison.png"), bbox_inches="tight")
plt.savefig(os.path.join(DRIVE_FIG_DIR, "qualitative_comparison.pdf"), bbox_inches="tight")
plt.show()
print(f"Saved qualitative results figure to: {qual_png} and Drive!")

# =====================================================================
# 4. PHYSICAL GROUND AREA QUANTIFICATION (0.5m GSD ANALYSIS)
# =====================================================================
print("\n--- 3. Running Physical Ground Area Quantification (Section 2.6) ---")
GSD = 0.5 # Ground Sampling Distance: 0.5 meters / pixel
PIXEL_AREA_M2 = GSD * GSD # 0.25 m^2 per pixel

area_results = []
total_pred_pixels = 0
total_gt_pixels = 0
total_parcels = 0

for fn in val_files[:500]: # Evaluate over 500 validation scenes
    t1_p = os.path.join(PATCHES_DIR, "train", "t1", fn)
    t2_p = os.path.join(PATCHES_DIR, "train", "t2", fn)
    lbl_p = os.path.join(PATCHES_DIR, "train", "label", fn)
    
    t1_t = normalize(T.ToTensor()(Image.open(t1_p).convert("RGB").resize((256, 256)))).unsqueeze(0).to(device)
    t2_t = normalize(T.ToTensor()(Image.open(t2_p).convert("RGB").resize((256, 256)))).unsqueeze(0).to(device)
    lbl_np = (np.array(Image.open(lbl_p).convert("L").resize((256, 256))) > 127).astype(np.uint8)
    
    with torch.no_grad():
        p = model(t1_t, t2_t)
        pred_bin = (torch.sigmoid(p).squeeze().cpu().numpy() > 0.5).astype(np.uint8)
        
    # Morphological noise opening (Paper Eq. 20)
    kernel = np.ones((3, 3), np.uint8)
    pred_clean = cv2.morphologyEx(pred_bin, cv2.MORPH_OPEN, kernel)
    
    # Connected component analysis for discrete parcels
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(pred_clean, connectivity=8)
    parcels_in_tile = max(0, num_labels - 1) # Exclude background label 0
    
    pred_pixels = np.sum(pred_clean)
    gt_pixels = np.sum(lbl_np)
    
    total_pred_pixels += pred_pixels
    total_gt_pixels += gt_pixels
    total_parcels += parcels_in_tile

# Compute physical area values
pred_area_m2 = total_pred_pixels * PIXEL_AREA_M2
pred_area_ha = pred_area_m2 / 10000.0
pred_area_km2 = pred_area_m2 / 1000000.0

gt_area_m2 = total_gt_pixels * PIXEL_AREA_M2
gt_area_ha = gt_area_m2 / 10000.0
gt_area_km2 = gt_area_m2 / 1000000.0

area_error_pct = abs(pred_area_m2 - gt_area_m2) / (gt_area_m2 + 1e-6) * 100

summary_df = pd.DataFrame({
    "Metric Parameter": [
        "Ground Sampling Distance (GSD)",
        "Single Pixel Unit Area",
        "Total Evaluated Validation Tiles",
        "Total Discrete Changed Parcels",
        "Predicted Total Changed Area (m²)",
        "Predicted Total Changed Area (Hectares)",
        "Predicted Total Changed Area (km²)",
        "Ground Truth Changed Area (Hectares)",
        "Area Measurement Relative Error (%)"
    ],
    "Quantified Value": [
        f"{GSD} m / pixel",
        f"{PIXEL_AREA_M2} m²",
        "500 patches (256x256)",
        f"{total_parcels:,} parcels",
        f"{pred_area_m2:,.2f} m²",
        f"{pred_area_ha:,.2f} ha",
        f"{pred_area_km2:,.4f} km²",
        f"{gt_area_ha:,.2f} ha",
        f"{area_error_pct:.2f} %"
    ]
})

print("\n=================== Physical Area Quantification Summary ===================")
print(summary_df.to_string(index=False))
print("============================================================================\n")

# Save table as CSV and LaTeX table
csv_path = os.path.join(LOCAL_FIG_DIR, "area_quantification_summary.csv")
summary_df.to_csv(csv_path, index=False)
summary_df.to_csv(os.path.join(DRIVE_FIG_DIR, "area_quantification_summary.csv"), index=False)
print(f"Area report saved to: {csv_path} and Drive!")
